# Read pdfs

This first part doesn't use AI yet — it's just normal Python file handling. A PDF can't be searched or "embedded" directly, so first we need to pull the plain text out of each PDF and save it as a `.txt` file. This cell is basically a copy-paste of what the real script `pdfs_to_text.py` does, so you can experiment with it live, one step at a time, before trusting it as a script.

In [1]:
from pypdf import PdfReader

def extract_text_from_pdf(path) -> str:
    reader = PdfReader(path)

    all_text = ""

    for page in reader.pages:
        text = page.extract_text()
        if text:
            all_text += text + "\n"

    return all_text

def export_text_to_txt(text, export_path):
    with open(export_path, "w") as file:
        file.write(text)


 

`Path` comes from Python's built-in `pathlib` module — it's a nicer way to represent a folder or file location than a plain string. Here we point it at the `data` folder and call `.glob("*.pdf")`, which means "list every file in this folder whose name ends in `.pdf`". In the real backend code this same folder is written once as `DATA_PATH` inside `constants.py`, so every script that needs it just imports that instead of retyping `Path("data")`.

In [2]:
from pathlib import Path 

data_path = Path("data")

list(data_path.glob("*.pdf"))

[WindowsPath('data/Dwarf_Hotot.pdf'),
 WindowsPath('data/Dwarf_rabbit.pdf'),
 WindowsPath('data/Holland_Lop.pdf'),
 WindowsPath('data/Netherland_Dwarf_rabbit.pdf')]

Now we actually do the conversion: for each PDF found above, call the function from the first cell to pull out its text, then save that text as a `.txt` file with a matching name (e.g. `holland_lop.pdf` → `holland_lop.txt`). This loop is the same code that runs automatically inside `pdfs_to_text.py`'s `if __name__ == "__main__":` block — that phrase just means "only run this part when the file is executed directly, not when something else imports it".

In [3]:

for pdf_path in data_path.glob("*.pdf"):
    pdf_text = extract_text_from_pdf(pdf_path)

    filename = f"{pdf_path.stem.casefold()}.txt"

    export_text_to_txt(pdf_text, data_path / filename)



**Before moving on:** there's a step happening *outside* this notebook that we're skipping over here. A separate script, `ingestion.py`, takes the `.txt` files we just created and turns each one's text into a list of numbers called an **embedding** — a way of representing the *meaning* of text as numbers, so a computer can later compare how similar two pieces of text are. It then saves everything into a small database on disk. You'd normally run that script once (`python -m backend.ingestion`) after this notebook finishes. Below, we just connect to the database it already built and look at what's inside — we're not building it here.

## Explore knowledge base

**LanceDB** is the vector database this project uses — think of it like a regular database (e.g. SQLite), but built specifically to store and search lists of numbers (embeddings) quickly. Here we connect to the database folder and call `.table_names()` just to confirm the `"articles"` table that `ingestion.py` created actually exists.

In [4]:
import lancedb 
from backend.constants import VECTOR_DATABASE_PATH

db = lancedb.connect(VECTOR_DATABASE_PATH)
db.table_names()

C:\Users\adelo\AppData\Local\Temp\ipykernel_26908\2949489873.py:5: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['articles']

`.to_pandas()` loads the whole table into a pandas DataFrame so we can look at it like a spreadsheet. Each row is one rabbit-breed document. The column names come from the `Article` schema in `data_models.py` — a **schema** is just a fixed list of column names and types that every row has to follow. The `embedding` column is the interesting one: it's that long list of numbers representing the text's meaning. We never typed those numbers ourselves — LanceDB generated them automatically from the `content` column when the row was added.

In [5]:
db["articles"].to_pandas()

,doc_id,filepath,filename,content,embedding
0,dwarf_hotot,C:\Users\adelo\de25\ai_ops_lecture\11_rag\data...,dwarf_hotot,Dwarf Hotot\nCountry\nof origin\nFrance\nUse S...,"[-0.036104, 0.007897, 0.021621, -0.049781, 0.0..."
1,dwarf_hotot,C:\Users\adelo\de25\ai_ops_lecture\11_rag\data...,dwarf_hotot,Dwarf Hotot\nCountry\nof origin\nFrance\nUse S...,"[-0.036104, 0.007897, 0.021621, -0.049781, 0.0..."
2,dwarf_rabbit,C:\Users\adelo\de25\ai_ops_lecture\11_rag\data...,dwarf_rabbit,A dwarf rabbit\n(Netherland Dwarf breed)\nBaby...,"[-0.015504, -0.005341, 0.023792, -0.057422, 0...."
3,holland_lop,C:\Users\adelo\de25\ai_ops_lecture\11_rag\data...,holland_lop,Holland Lop\nHolland lop standing on her back ...,"[-0.025575, 0.011408, 0.012832, -0.056407, 0.0..."
4,netherland_dwarf_rabbit,C:\Users\adelo\de25\ai_ops_lecture\11_rag\data...,netherland_dwarf_rabbit,Netherland Dwarf rabbit\nCountry of\norigin\n ...,"[-0.010192, 0.002978, 0.016219, -0.05453, 0.01..."


`.schema` shows the table's structure directly, without loading all the data. Notice `embedding: fixed_size_list<item: float>[3072]` — that `3072` is how many numbers are in each embedding, and it has to exactly match `EMBEDDING_DIM = 3072` in `data_models.py`, because that's how many numbers Google's `gemini-embedding-001` model always produces for a piece of text. The `embedding_functions` metadata underneath tells LanceDB "use this same embedding model later when someone searches the table", so a search query gets turned into numbers the same way the stored documents were — otherwise comparing them wouldn't make sense.

In [6]:
db["articles"].schema

doc_id: string not null
filepath: string not null
filename: string not null
content: string not null
embedding: fixed_size_list<item: float>[3072]
  child 0, item: float
-- schema metadata --
embedding_functions: '[
  {
    "name": "gemini-text",
    "model": {
   ' + 130